In [0]:
pip install pytrends

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Ingest Google Trends (Production)
# MAGIC 
# MAGIC **MODE = incremental** (default, scheduled):
# MAGIC - Skips if last run was < 7 days ago (weekly data, no point running daily)
# MAGIC - Always uses "today 5-y" (Google requires this for consistent normalization)
# MAGIC - Deletes old files per keyword after writing new ones (keeps storage flat)
# MAGIC 
# MAGIC **MODE = backfill** (manual):
# MAGIC - Ignores skip logic — always runs
# MAGIC - Deletes old files per keyword (clean slate)
# MAGIC - Same data pull (Google Trends has no date-range incremental API)

# COMMAND ----------



# COMMAND ----------

import json
import datetime as dt
import pandas as pd
from typing import Dict, Any
from pytrends.request import TrendReq

# COMMAND ----------

# ---------------- Config ----------------
CATALOG = "canada_business"
SCHEMA  = "bronze"
BRONZE_SUBDIR = "google_trends_raw"

GEO = "CA"
KEYWORDS = ["buy laptop", "TV deals", "gaming console", "electronics sale"]
TIMEFRAME = "today 5-y"
MIN_DAYS_BETWEEN_RUNS = 7

# COMMAND ----------

# ---------------- Mode ----------------
dbutils.widgets.dropdown("MODE", "incremental", ["incremental", "backfill"])
MODE = dbutils.widgets.get("MODE").strip().lower()
print(f"MODE: {MODE}")

# COMMAND ----------

# ---------------- Helpers ----------------
def run_ts_utc() -> str:
    return dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def resolve_schema_root_location(catalog: str, schema: str) -> str:
    df = spark.sql(f"DESCRIBE SCHEMA EXTENDED {catalog}.{schema}").toPandas()
    rows = df.loc[df["database_description_item"] == "RootLocation", "database_description_value"].values
    if len(rows) == 0 or not rows[0]:
        raise RuntimeError(f"RootLocation not found for {catalog}.{schema}")
    return str(rows[0])

def join_uri(base_uri: str, child: str) -> str:
    return base_uri.rstrip("/") + "/" + child.strip("/")

# COMMAND ----------

# ---------------- Skip check (incremental only) ----------------
if MODE == "incremental":
    last_run_df = spark.sql("""
        SELECT MAX(last_run_ts) AS last_ts
        FROM canada_business.bronze.ingest_watermarks
        WHERE source = 'google_trends'
    """).collect()

    last_ts = last_run_df[0]["last_ts"]
    if last_ts is not None:
        last_run_date = dt.datetime.strptime(last_ts[:8], "%Y%m%d").date()
        days_since = (dt.date.today() - last_run_date).days
        if days_since < MIN_DAYS_BETWEEN_RUNS:
            print(f"⏭️ Last run was {days_since} days ago (< {MIN_DAYS_BETWEEN_RUNS}). Skipping.")
            dbutils.notebook.exit(f"SKIPPED: last run {days_since} days ago")
        else:
            print(f"Last run was {days_since} days ago. Proceeding.")
    else:
        print("No previous run found. Running initial ingest.")
else:
    print("Backfill mode — skipping recency check.")

# COMMAND ----------

# ---------------- Resolve paths ----------------
root_location = resolve_schema_root_location(CATALOG, SCHEMA)
bronze_base = join_uri(root_location, BRONZE_SUBDIR)
data_base = bronze_base.rstrip("/") + "/data"
manifest_base = bronze_base.rstrip("/") + "/manifests"

dbutils.fs.mkdirs(data_base)
dbutils.fs.mkdirs(manifest_base)

# COMMAND ----------

# ---------------- Fetch from Google Trends ----------------
ts = run_ts_utc()
print(f"Run ts: {ts} | MODE: {MODE}")

pytrends = TrendReq(hl="en-CA", tz=0)
pytrends.build_payload(KEYWORDS, timeframe=TIMEFRAME, geo=GEO)

df = pytrends.interest_over_time().reset_index()
if "isPartial" in df.columns:
    df = df.drop(columns=["isPartial"])

print(f"Fetched {len(df)} weeks of data")

manifest: Dict[str, Any] = {
    "run_ts": ts,
    "mode": MODE,
    "geo": GEO,
    "timeframe": TIMEFRAME,
    "keywords": KEYWORDS,
    "files": []
}

# COMMAND ----------

# ---------------- Write new files + cleanup old ones ----------------
for kw in KEYWORDS:
    kw_dir = kw.replace(" ", "_").lower()
    out_dir = join_uri(data_base, kw_dir)
    dbutils.fs.mkdirs(out_dir)

    new_filename = f"trend_{ts}.json"
    out_file = join_uri(out_dir, new_filename)

    payload = {
        "keyword": kw,
        "geo": GEO,
        "timeframe": TIMEFRAME,
        "run_ts": ts,
        "observations": [
            {"date": str(r["date"].date()), "value": float(r[kw]) if pd.notna(r[kw]) else None}
            for _, r in df.iterrows()
        ]
    }

    # Write new file
    dbutils.fs.put(out_file, json.dumps(payload, indent=2), overwrite=False)
    obs_count = len(payload["observations"])
    print(f"✅ {kw} -> {obs_count} obs -> {out_file}")

    # Delete old files (keep only the new one)
    try:
        for f in dbutils.fs.ls(out_dir):
            if f.name != new_filename:
                dbutils.fs.rm(f.path)
                print(f"  🗑️ Deleted: {f.name}")
    except Exception as e:
        print(f"  ⚠️ Cleanup warning: {e}")

    manifest["files"].append({
        "keyword": kw, "rows": obs_count, "raw_file": out_file
    })

    # Update watermark
    max_date = max(o["date"] for o in payload["observations"] if o["date"])
    spark.sql(f"""
        MERGE INTO canada_business.bronze.ingest_watermarks AS t
        USING (SELECT
            'google_trends' AS source, '{kw}' AS series_key,
            DATE('{max_date}') AS last_obs_date, '{ts}' AS last_run_ts,
            {obs_count} AS rows_written, current_timestamp() AS updated_at
        ) AS s
        ON t.source = s.source AND t.series_key = s.series_key
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

# Write manifest
manifest_path = join_uri(manifest_base, f"run_manifest_{ts}.json")
dbutils.fs.put(manifest_path, json.dumps(manifest, indent=2), overwrite=False)

print(f"\n✅ Google Trends ingest complete ({MODE})")
print("Manifest:", manifest_path)